In [ ]:
!pip install -q groq

import asyncio
import os
import json
from groq import AsyncGroq

# Replace with your actual Groq API Key
os.environ["GROQ_API_KEY"] = "your-api-key-here"

client = AsyncGroq(api_key=os.environ.get("GROQ_API_KEY"))

print("✅ Environment Ready: Groq Client Initialized.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 6.0 MB/s eta 0:00:00
✅ Environment Ready: Groq Client Initialized.


In [ ]:
async def call_llm(model, role_name, prompt, system_prompt):
    """Helper to call models and display progress in real-time."""
    print(f"\n--- [LOG] {role_name} is thinking... ({model}) ---")

    try:
        completion = await client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            model=model,
        )
        response = completion.choices[0].message.content
        print(f"--- [LOG] {role_name} Finished ---")
        return response
    except Exception as e:
        print(f"❌ Error in {role_name}: {e}")
        return f"FAILED: {e}"

# --- DSE SYSTEM PROMPTS ---

PROPOSER_1_SYSTEM = """You are Lead Architect A.
Focus: Industry-standard robustness, high performance, and long-term sustainability.
Propose a complete tech stack and logical architecture for the problem statement."""

PROPOSER_2_SYSTEM = """You are Lead Architect B.
Focus: Maximum cost-efficiency, speed of development, and modern open-source/serverless tools.
Propose a complete tech stack and logical architecture for the problem statement."""

CRITIC_SYSTEM = """You are a Destructive Critic.
Your only job is to find failure modes, scaling cliffs, and security holes in the provided proposals.
MANDATORY: Provide a score (1-10) for each proposal based on Feasibility, Cost, and Risk.
Do not suggest solutions; only identify weaknesses."""

ARBITER_SYSTEM = """You are the Authoritative Arbiter.
Review two proposals and the critic's analysis.
You must choose EXACTLY one architecture. No merging allowed.
Provide a clear, authoritative rationale for your decision."""

In [ ]:
async def execute_dse_workflow(problem, constraints):
    # Prepare the context
    context = f"Problem: {problem}\nConstraints: {constraints if constraints else 'Use Market Standard (Cheap/Sustainable)'}"

    print("\n🚀 STEP 1: PARALLEL PROPOSAL GENERATION")
    # Proposer 1 (70B) and Proposer 2 (8B) start at the exact same time
    p1_task = call_llm("llama-3.3-70b-versatile", "Proposer 1", context, PROPOSER_1_SYSTEM) #Put the highest billion parameter models for both proposers.
    p2_task = call_llm("llama-3.1-8b-instant", "Proposer 2", context, PROPOSER_2_SYSTEM)

    # Wait for both to complete
    p1_res, p2_res = await asyncio.gather(p1_task, p2_task)

    print("\n" + "="*60)
    print(f"PROPOSER 1 OUTPUT (Robust):\n{p1_res}")
    print("-" * 60)
    print(f"PROPOSER 2 OUTPUT (Frugal):\n{p2_res}")
    print("="*60)

    print("\n🔨 STEP 2: SEQUENTIAL CRITIC ATTACK")
    critic_prompt = f"Analyze and score these architectures:\n\nP1: {p1_res}\n\nP2: {p2_res}"
    critic_res = await call_llm("openai/gpt-oss-20b", "Critic", critic_prompt, CRITIC_SYSTEM) #Okay but do evaluate models before selecting. Use Poetiq AGI model.
    print(f"\nCRITIC REPORT:\n{critic_res}")

    print("\n⚖️ STEP 3: SEQUENTIAL ARBITER VERDICT")
    arb_prompt = f"Proposals:\nP1: {p1_res}\nP2: {p2_res}\n\nCritique & Scores:\n{critic_res}"
    arb_res = await call_llm("qwen-qwq-32b", "Arbiter", arb_prompt, ARBITER_SYSTEM) #Okay but do evaluate models before selecting

    print(f"\nFINAL ARBITER DECISION:\n{arb_res}")

    return {
        "problem": problem,
        "p1": p1_res,
        "p2": p2_res,
        "critic": critic_res,
        "arbiter": arb_res
    }

In [ ]:
import time # Added this to prevent the NameError

# User Interaction
problem_stmt = input("Describe the problem statement: ")
budget = input("Budget (₹/$): ")
timeline = input("Time span: ")
tools = input("Specific tools to integrate (if any): ")

user_constraints = {
    "budget": budget,
    "time": timeline,
    "tools": tools
}

# Run DSE Engine
# We use 'await' because execute_dse_workflow is an async function
results = await execute_dse_workflow(problem_stmt, user_constraints)

# Logging the results
print("\n💾 [STORAGE] Indexing project session in Database...")

data_to_save = {
    "timestamp": time.ctime(), # This will now work!
    "problem": results['problem'],
    "p1_architecture": results['p1'],
    "p2_architecture": results['p2'],
    "critic_score": results['critic'],
    "final_selection": results['arbiter']
}

# In a real app, you'd insert 'data_to_save' into MongoDB or PostgreSQL here
print("✅ Project history saved successfully.")
print(f"Timestamp of entry: {data_to_save['timestamp']}")

Describe the problem statement: write a web application for calculating fibonacci series based on initial input number from the user
Budget (₹/$): 20000
Time span: 6 days
Specific tools to integrate (if any): streamlit application

🚀 STEP 1: PARALLEL PROPOSAL GENERATION

--- [LOG] Proposer 1 is thinking... (llama-3.3-70b-versatile) ---

--- [LOG] Proposer 2 is thinking... (llama-3.1-8b-instant) ---
--- [LOG] Proposer 2 Finished ---
--- [LOG] Proposer 1 Finished ---

PROPOSER 1 OUTPUT (Robust):
**Tech Stack and Logical Architecture for Fibonacci Series Calculator**

Given the constraints of a $20,000 budget, 6-day timeline, and the requirement to use Streamlit for the application, I propose the following tech stack and logical architecture for the Fibonacci series calculator:

### Tech Stack

* **Frontend:** Streamlit (Python-based library for building web applications)
* **Backend:** Python 3.9+ (for calculation and data processing)
* **Database:** None required (calculations can be pe